# الفصل الرابع والعشرون: كتابة كود نظيف (Clean Code)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-24.ipynb)

> **الوقت المتوقع:** ~9 دقيقة قراءة · **12** قسمًا · **27** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **24.1** لماذا «الكود الذي يعمل» لا يكفي؟
- **24.2** PEP 8: دليل الأسلوب الرسمي لبايثون
- **24.3** التنسيق الآلي: `black`
- **24.4** الفحص الآلي للأخطاء والجودة: `ruff`
- **24.5** مبادئ أساسية لكتابة دوال وأصناف نظيفة
- **24.6** مقدمة مبسّطة لمبادئ SOLID
- **24.7** ممارسات إضافية للكود النظيف
- **24.8** نموذج عملي: قبل وبعد (Refactoring كامل)
- **24.9** تمارين
- **24.10** اختبر فهمك (Quiz)
- **24.11** ملخص الفصل (Cheat Sheet)
- **24.12** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["black", "ruff"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q black ruff

> ℹ️ أدوات التنسيق تُشغَّل كأوامر خارجية عبر `!black` و`!ruff`.

## 24.1 لماذا «الكود الذي يعمل» لا يكفي؟

حتى الآن، ركّزنا على هدف واحد رئيسي: **أن يعمل الكود ويُنتج النتيجة الصحيحة**. لكن هناك سؤال آخر لا يقل أهمية: **هل يستطيع إنسان آخر (أو أنتَ نفسك بعد 6 أشهر) أن يفهم هذا الكود بسهولة؟**

تخيّل هذا الموقف الشائع جدًا في حياة كل مبرمج: تفتح ملفًا كتبته بنفسك قبل نصف عام، لحل مشكلة بسيطة فيه، فتجد نفسك تحدّق في الشاشة لعشر دقائق محاولًا فهم «ماذا كنتُ أفكر هنا؟!». هذا ليس نقصًا في الذكاء — إنه نتيجة طبيعية لكتابة **كود غير نظيف**.

> 💭 **مقولة شهيرة في عالم البرمجة:**
> *«اكتب الكود وكأن الشخص الذي سيصونه هو قاتل مختل يعرف أين تسكن.»* — مطوّر مجهول (نصيحة مبالغ فيها، لكن الفكرة واضحة: اجعل كودك مفهومًا بلا جهد)

**الكود النظيف (Clean Code)** ليس ترفًا أو مثالية زائدة — إنه ما يُفرّق بين مبرمج هاوٍ ومبرمج محترف. الشركات الكبرى لا تُقيّم المبرمجين فقط بناءً على «هل يحل المشكلة؟»، بل «هل يحلها بطريقة واضحة، قابلة للصيانة، ولا تُسبب كوارث لاحقة؟».

في هذا الفصل، سنتعلّم:
1. **PEP 8**: دليل الأسلوب الرسمي للغة بايثون.
2. أدوات الفحص والتنسيق الآلي: **`black`** و **`ruff`**.
3. مبادئ **SOLID** المبسّطة لتصميم كود قابل للتوسّع والصيانة.
4. ممارسات عملية لتسمية المتغيرات، تنظيم الدوال، وكتابة تعليقات مفيدة.

## 24.2 PEP 8: دليل الأسلوب الرسمي لبايثون

**PEP** تعني "Python Enhancement Proposal" (اقتراح تحسين بايثون) — وهي وثائق رسمية تصف تغييرات أو معايير للغة. **PEP 8** تحديدًا هي الوثيقة الأشهر على الإطلاق: دليل الأسلوب القياسي لكتابة كود بايثون. لقد كنا نتبع معظم قواعده طوال الكتاب دون أن نسمّيه صراحة — الآن حان وقت التوثيق الرسمي.

### قواعد التسمية (Naming Conventions)

| النوع | الأسلوب | مثال |
|---|---|---|
| المتغيرات والدوال | `snake_case` | `user_name`, `calculate_total()` |
| الثوابت | `UPPER_SNAKE_CASE` | `MAX_RETRIES`, `PI` |
| الأصناف (Classes) | `PascalCase` | `UserAccount`, `TaskManager` |
| الخاصيات المحمية | `_single_leading_underscore` | `_internal_cache` |
| الخاصيات الخاصة | `__double_leading_underscore` | `__secret_key` |
| الوحدات (ملفات .py) | `snake_case` قصير | `data_loader.py` |
| أسماء غير مستخدمة | `_` أو `_unused` | `for _ in range(10):` |

> **⚠️ تحذير: تجنّب الأسماء المضلِّلة أو المبهمة**
> ```python
> # ❌ سيء جدًا -- لا يخبرنا بشيء
> def calc(x, y, z):
>     return x * y - z
>
> # ✅ ممتاز -- الاسم نفسه يشرح الغرض
> def calculate_net_price(unit_price: float, quantity: int, discount: float) -> float:
>     return unit_price * quantity - discount
> ```
> الكود الجيد **يُقرأ كالنثر**. إذا احتجت تعليقًا لشرح ماذا تفعل دالة أو متغيّر، فكّر أولًا: هل يمكن أن يشرح الاسم نفسه هذا بدلاً من التعليق؟

### المسافات والتباعد

```python
# ✅ صحيح: مسافة واحدة حول عوامل التشغيل
total = price + tax

# ❌ خطأ: بدون مسافات، أو مسافات زائدة
total=price+tax
total  =  price  +  tax

# ✅ صحيح: بدون مسافة قبل الفاصلة، مسافة بعدها
def greet(name, age):
    pass

# ✅ صحيح: بدون مسافات حول = في المعاملات الافتراضية (دون type hint)
def greet(name="أحمد"):
    pass

# ✅ لكن مع type hint، نضيف مسافات حول =
def greet(name: str = "أحمد"):
    pass
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### طول السطر

PEP 8 يوصي بحد أقصى **79 حرفًا** للسطر تقليديًا، لكن المعيار الشائع اليوم في المشاريع الحديثة هو **88 حرفًا** (وهو الافتراضي في أداة `black` التي سنتعلمها بعد قليل). الفكرة: الأسطر الطويلة جدًا يصعب قراءتها، خصوصًا عند عرض ملفين جنبًا إلى جنب.

```python
# ❌ سطر طويل جدًا، يصعب قراءته دفعة واحدة
result = calculate_total_price(unit_price, quantity, discount_percentage, tax_rate, shipping_cost)

# ✅ مقسّم بشكل مقروء
result = calculate_total_price(
    unit_price,
    quantity,
    discount_percentage,
    tax_rate,
    shipping_cost,
)
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### الاستيراد (Imports)

```python
# ✅ الترتيب الموصى به: مكتبات قياسية، ثم مكتبات خارجية، ثم وحدات المشروع
import json
import os
from pathlib import Path

import pandas as pd
import requests

from my_project.models import User
from my_project.utils import format_date

# ❌ تجنّب الاستيراد الشامل -- يُخفي مصدر الأسماء ويُسبب تعارضات
from os import *

# ✅ استورد ما تحتاجه فقط بوضوح
from os import getcwd, path
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> **💡 نصيحة خبير:** لا تحتاج لحفظ كل قواعد PEP 8 عن ظهر قلب — هذا بالضبط ما تفعله الأدوات الآلية التي سنتعلمها الآن تلقائيًا. المهم أن تفهم **الفلسفة**: الوضوح، الاتساق، والقابلية للقراءة.

## 24.3 التنسيق الآلي: `black`

بدلاً من الجدال حول «أين أضع الفاصلة؟» أو «هل أستخدم علامات اقتباس مفردة أم مزدوجة؟»، تحلّ أداة **`black`** هذا الجدال نهائيًا: إنها «منسّق عنيد» (Opinionated Formatter) يُعيد كتابة كودك تلقائيًا بأسلوب موحّد وثابت، بلا نقاش.

### التثبيت والاستخدام

```bash
uv add --dev black
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

In [ ]:
# قبل black (ملف messy.py)
def   calculate(x,y) :
    result=x+y
    if result>10:
        return   'كبير'
    else:
        return "صغير"

```bash
uv run black messy.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

In [ ]:
# بعد black -- تلقائيًا!
def calculate(x, y):
    result = x + y
    if result > 10:
        return "كبير"
    else:
        return "صغير"

لاحظ: `black` وحّد علامات الاقتباس إلى مزدوجة، أصلح المسافات، ونظّم كل شيء — **دون أن تكتب قاعدة واحدة بنفسك**.

```bash
# فحص الملفات بدون تعديلها فعليًا (مفيد في CI/CD)
uv run black --check .

# معاينة التغييرات المقترحة دون تطبيقها
uv run black --diff .

# تنسيق كل ملفات المشروع
uv run black .
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

> **💡 نصيحة خبير:** اضبط VS Code (الإضافة التي ثبّتناها في الفصل 2) على «التنسيق عند الحفظ» (Format on Save) باستخدام `black`. بهذا، كل مرة تحفظ فيها ملفًا، يُنسَّق تلقائيًا دون أي جهد واعٍ منك.

> **🔄 مقارنة: لماذا «عنيد» (Opinionated) أمر جيد؟**
>
> | أداة تنسيق مرنة (إعدادات كثيرة) | `black` (عنيد، إعدادات قليلة) |
> |---|---|
> | كل فريق يختار أسلوبًا مختلفًا | أسلوب موحّد عالميًا بين كل مشاريع بايثون |
> | نقاشات لا تنتهي في مراجعات الكود حول الأسلوب | لا نقاش — الأداة تقرر |
> | وقت ضائع في التنسيق اليدوي | وقت التنسيق = صفر |
>
> هذا بالضبط سبب نجاح `black`: إنه يُنهي «حرب الأساليب» بشكل نهائي.

## 24.4 الفحص الآلي للأخطاء والجودة: `ruff`

بينما `black` يُنسّق **شكل** الكود، أداة **`ruff`** (Linter) تفحص **جودة ومنطق** الكود: متغيرات غير مستخدمة، استيراد لا داعي له، أخطاء منطقية محتملة، ومخالفات PEP 8 التي لا يُصلحها `black`.

> **🔄 مقارنة: Formatter مقابل Linter**
>
> | | Formatter (مثل `black`) | Linter (مثل `ruff`) |
> |---|---|---|
> | يهتم بـ | الشكل (مسافات، أسطر، اقتباسات) | المحتوى (أخطاء محتملة، كود ميت، أنماط سيئة) |
> | مثال | إعادة ترتيب المسافات | تحذير: "المتغيّر `x` مُعرَّف لكن غير مستخدم أبدًا" |
> | يُعدّل الكود تلقائيًا؟ | نعم، دائمًا | أحيانًا (عبر `--fix`) |

### لماذا `ruff` تحديدًا؟

`ruff` أداة حديثة مكتوبة بلغة Rust، وأصبحت المعيار الفعلي (de facto standard) في مجتمع بايثون الحديث لأنها **أسرع بعشرات المرات** من الأدوات التقليدية (مثل `flake8` و`pylint`)، وتجمع وظائف عدة أدوات قديمة في أداة واحدة سريعة.

### التثبيت والاستخدام

```bash
uv add --dev ruff
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

In [ ]:
# problematic.py
import os  # مستورد لكن غير مستخدم
import sys

def process_data(data):
    unused_variable = 42  # معرّف لكن غير مستخدم
    result = []
    for item in data:
        if item == None:  # يجب استخدام "is None" وليس "== None"
            continue
        result.append(item)
    return result

```bash
uv run ruff check problematic.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
problematic.py:1:8: F401 `os` imported but unused
problematic.py:8:5: F841 local variable `unused_variable` is assigned to but never used
problematic.py:11:16: E711 comparison to None should be 'if cond is None:'
Found 3 errors.
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

```bash
# إصلاح تلقائي لما يمكن إصلاحه آليًا
uv run ruff check --fix problematic.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### إعدادات `ruff` في `pyproject.toml`

```toml
[tool.ruff]
line-length = 88
target-version = "py314"

[tool.ruff.lint]
select = [
    "E",   # pycodestyle errors
    "F",   # pyflakes (أخطاء منطقية)
    "I",   # ترتيب الاستيراد (isort)
    "N",   # قواعد تسمية PEP 8
    "UP",  # ترقية تلقائية لصياغات بايثون حديثة
]
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

> **💡 نصيحة خبير:** شغّل `black` و`ruff` معًا دائمًا كجزء من روتين عملك اليومي:
> ```bash
> uv run ruff check --fix .   # إصلاح المشاكل المنطقية أولاً
> uv run black .               # تنسيق الشكل أخيرًا
> ```
> هذا الثنائي (`black` + `ruff`) هو المعيار شبه الرسمي في معظم مشاريع بايثون الاحترافية الحديثة اليوم.

## 24.5 مبادئ أساسية لكتابة دوال وأصناف نظيفة

### مبدأ المسؤولية الواحدة (دالة واحدة = مهمة واحدة)

```python
# ❌ سيء: دالة واحدة تفعل أشياء كثيرة جدًا
def process_order(order_data):
    # التحقق من الصحة
    if not order_data.get("items"):
        raise ValueError("لا توجد عناصر")

    # حساب الإجمالي
    total = sum(item["price"] * item["qty"] for item in order_data["items"])

    # تطبيق الخصم
    if order_data.get("coupon"):
        total *= 0.9

    # حفظ في قاعدة البيانات
    print(f"حفظ الطلب بإجمالي {total}")

    # إرسال بريد إلكتروني
    print("إرسال تأكيد عبر البريد...")

    return total

# ✅ جيد: كل دالة مسؤولة عن شيء واحد فقط
def validate_order(order_data: dict) -> None:
    """التحقق من صحة بيانات الطلب."""
    if not order_data.get("items"):
        raise ValueError("لا توجد عناصر في الطلب")

def calculate_total(items: list[dict]) -> float:
    """حساب إجمالي سعر العناصر."""
    return sum(item["price"] * item["qty"] for item in items)

def apply_discount(total: float, has_coupon: bool) -> float:
    """تطبيق خصم 10% إذا وُجد كوبون."""
    return total * 0.9 if has_coupon else total

def save_order(total: float) -> None:
    """حفظ الطلب في قاعدة البيانات."""
    print(f"حفظ الطلب بإجمالي {total}")

def send_confirmation_email() -> None:
    """إرسال بريد تأكيد الطلب."""
    print("إرسال تأكيد عبر البريد...")

def process_order(order_data: dict) -> float:
    """تنسيق عملية معالجة الطلب الكاملة."""
    validate_order(order_data)
    total = calculate_total(order_data["items"])
    total = apply_discount(total, bool(order_data.get("coupon")))
    save_order(total)
    send_confirmation_email()
    return total
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> **💡 لماذا هذا أفضل؟**
> - كل دالة **قابلة للاختبار بمفردها** (سنتعلّم `pytest` في الفصل القادم).
> - يسهل **إعادة استخدام** أي دالة في سياق آخر (مثلاً `calculate_total` في تقرير مختلف).
> - عند حدوث خطأ، تعرف **بالضبط** أين تبحث.
> - القراءة من `process_order` تُشبه قراءة فهرس كتاب: واضحة ومباشرة.

### تجنّب «الأرقام السحرية» (Magic Numbers) والنصوص الحرفية المتكررة

```python
# ❌ سيء: ما معنى 0.15؟ ولماذا 3؟
def calculate_price(amount):
    if amount > 1000:
        return amount * 0.85
    return amount * 0.95

for attempt in range(3):
    try_connection()

# ✅ جيد: الثوابت المُسمّاة تشرح نفسها
BULK_DISCOUNT_THRESHOLD = 1000
BULK_DISCOUNT_RATE = 0.85
STANDARD_DISCOUNT_RATE = 0.95
MAX_CONNECTION_ATTEMPTS = 3

def calculate_price(amount: float) -> float:
    if amount > BULK_DISCOUNT_THRESHOLD:
        return amount * BULK_DISCOUNT_RATE
    return amount * STANDARD_DISCOUNT_RATE

for attempt in range(MAX_CONNECTION_ATTEMPTS):
    try_connection()
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### Guard Clauses بدلاً من التداخل العميق

تعلمنا هذا في الفصل 6 و10، لكنه يستحق التذكير كممارسة من ممارسات الكود النظيف الأساسية:

```python
# ❌ سيء: تداخل عميق (Nested Ifs) صعب التتبع
def get_discount(user):
    if user is not None:
        if user.is_active:
            if user.membership == "premium":
                return 0.2
            else:
                return 0.1
        else:
            return 0
    else:
        return 0

# ✅ جيد: Guard Clauses -- خروج مبكر، أقل تداخلًا
def get_discount(user) -> float:
    if user is None:
        return 0
    if not user.is_active:
        return 0
    if user.membership == "premium":
        return 0.2
    return 0.1
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### التعليقات: متى تكتبها ومتى تتجنّبها

```python
# ❌ تعليق زائد لا يُضيف أي معلومة
x = x + 1  # إضافة 1 إلى x

# ❌ تعليق يشرح "ماذا" بدلاً من "لماذا" (الكود نفسه يوضّح "ماذا")
# نحسب السعر الإجمالي بضرب السعر في الكمية
total = price * quantity

# ✅ تعليق مفيد يشرح "لماذا" -- معلومة لا يمكن استنتاجها من الكود نفسه
# نستخدم هامش أمان 5% لتعويض تقلبات أسعار الصرف اليومية
total = price * quantity * 1.05

# ✅ تعليق يشرح قرارًا غير بديهي
# ملاحظة: نستخدم format التاريخ القديم هنا للتوافق مع النظام القديم (legacy system)
date_string = date.strftime("%d/%m/%Y")
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> **💡 نصيحة خبير ذهبية:** «التعليق الجيد يشرح **لماذا**، والكود الجيد يشرح **ماذا**.» إذا وجدت نفسك تكتب تعليقًا لشرح «ماذا يفعل هذا السطر»، فكّر أولًا في إعادة تسمية المتغيرات أو الدوال لتكون أوضح، بدلاً من الاعتماد على التعليق.

## 24.6 مقدمة مبسّطة لمبادئ SOLID

**SOLID** هي اختصار لخمسة مبادئ لتصميم كود كائني التوجه (OOP، كما تعلمنا في الفصلين 13 و14) قابل للصيانة والتوسّع. لن نتعمّق في كل التفاصيل الأكاديمية، بل نفهم **الفكرة العملية** وراء كل مبدأ.

### S — مبدأ المسؤولية الواحدة (Single Responsibility Principle)

**«يجب أن يكون لكل صنف (class) سبب واحد فقط للتغيير.»**

```python
# ❌ ينتهك SRP: صنف واحد يفعل أشياء كثيرة غير مرتبطة
class Report:
    def __init__(self, data: list[dict]):
        self.data = data

    def calculate_summary(self) -> dict:
        return {"total": sum(d["value"] for d in self.data)}

    def save_to_file(self, path: str) -> None:  # مسؤولية مختلفة: التخزين!
        with open(path, "w") as f:
            f.write(str(self.data))

    def send_email(self, address: str) -> None:  # مسؤولية مختلفة: الاتصال!
        print(f"إرسال التقرير إلى {address}")

# ✅ يتبع SRP: كل صنف مسؤول عن شيء واحد
class Report:
    def __init__(self, data: list[dict]):
        self.data = data

    def calculate_summary(self) -> dict:
        return {"total": sum(d["value"] for d in self.data)}

class ReportStorage:
    @staticmethod
    def save_to_file(report: Report, path: str) -> None:
        with open(path, "w") as f:
            f.write(str(report.data))

class ReportMailer:
    @staticmethod
    def send_email(report: Report, address: str) -> None:
        print(f"إرسال التقرير إلى {address}")
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**الفائدة العملية:** إذا تغيّرت طريقة الحفظ (من ملف إلى قاعدة بيانات)، تُعدّل `ReportStorage` فقط، دون المخاطرة بكسر منطق حساب التقرير أو الإرسال.

### O — مبدأ الفتح والإغلاق (Open/Closed Principle)

**«يجب أن يكون الكود مفتوحًا للتوسّع، مغلقًا للتعديل.»** أي: يمكنك إضافة سلوك جديد **دون تعديل الكود الموجود والمُختبَر مسبقًا**.

In [ ]:
from abc import ABC, abstractmethod

# ✅ يتبع OCP: إضافة طريقة دفع جديدة لا تتطلب تعديل أي كود قديم
class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str:
        ...

class CreditCardPayment(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"دفع {amount} عبر بطاقة الائتمان"

class PayPalPayment(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"دفع {amount} عبر PayPal"

# لإضافة طريقة دفع جديدة، أضف صنفًا جديدًا فقط -- بدون لمس الكود القديم!
class CryptoPayment(PaymentMethod):
    def pay(self, amount: float) -> str:
        return f"دفع {amount} عبر العملات الرقمية"

def checkout(method: PaymentMethod, amount: float) -> None:
    print(method.pay(amount))

checkout(CreditCardPayment(), 100)
checkout(CryptoPayment(), 50)

> هذا يُذكّرنا بـ Abstract Base Classes وتعدد الأشكال (Polymorphism) التي تعلمناها في الفصل 14 — وهي الأداة التقنية الأساسية لتطبيق هذا المبدأ.

### L — مبدأ استبدال ليسكوف (Liskov Substitution Principle)

**«يجب أن يكون بإمكان استبدال أي صنف فرعي بصنفه الأساسي دون كسر البرنامج.»**

```python
# ❌ ينتهك LSP: الصنف الفرعي يُغيّر السلوك المتوقع بشكل مفاجئ
class Bird:
    def fly(self) -> str:
        return "أطير في السماء"

class Penguin(Bird):
    def fly(self) -> str:
        raise NotImplementedError("البطريق لا يطير!")  # مفاجأة سيئة لأي كود يتوقع fly()

# ✅ يتبع LSP: إعادة تصميم الهرمية لتعكس الواقع بدقة
class Bird(ABC):
    @abstractmethod
    def move(self) -> str:
        ...

class FlyingBird(Bird):
    def move(self) -> str:
        return "أطير في السماء"

class Penguin(Bird):
    def move(self) -> str:
        return "أسبح في الماء"  # سلوك صحيح ومتوقع، بدون كسر توقعات المستخدم
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**الدرس العملي:** إذا وجدت نفسك تكتب `if isinstance(bird, Penguin): ... else: ...` للتعامل مع حالة استثنائية لصنف فرعي، فهذا مؤشر قوي على انتهاك LSP — عيد التفكير في التصميم الهرمي.

### I — مبدأ فصل الواجهات (Interface Segregation Principle)

**«لا تُجبر صنفًا على تطبيق دوال لا يحتاجها.»**

```python
# ❌ ينتهك ISP: واجهة ضخمة تُجبر كل شيء على تطبيق كل شيء
class Worker(ABC):
    @abstractmethod
    def work(self) -> str: ...
    @abstractmethod
    def eat(self) -> str: ...

class RobotWorker(Worker):
    def work(self) -> str:
        return "أعمل 24 ساعة"

    def eat(self) -> str:
        raise NotImplementedError("الروبوت لا يأكل!")  # إجبار غير منطقي

# ✅ يتبع ISP: واجهات صغيرة ومتخصصة، يطبّق كل صنف ما يحتاجه فقط
class Workable(ABC):
    @abstractmethod
    def work(self) -> str: ...

class Eatable(ABC):
    @abstractmethod
    def eat(self) -> str: ...

class HumanWorker(Workable, Eatable):
    def work(self) -> str:
        return "أعمل 8 ساعات"

    def eat(self) -> str:
        return "أتناول الغداء"

class RobotWorker(Workable):  # لا حاجة لتطبيق eat() أبدًا
    def work(self) -> str:
        return "أعمل 24 ساعة"
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### D — مبدأ انعكاس الاعتماديات (Dependency Inversion Principle)

**«اعتمد على التجريد (Abstractions)، وليس على التفاصيل الملموسة (Concrete Implementations).»**

```python
# ❌ ينتهك DIP: الصنف مرتبط مباشرة بتفصيل محدد (MySQL)
class UserRepository:
    def __init__(self):
        self.db = MySQLDatabase()  # اعتماد مباشر وصلب على تفصيل محدد

    def get_user(self, user_id: int):
        return self.db.query(f"SELECT * FROM users WHERE id={user_id}")

# ✅ يتبع DIP: الاعتماد على تجريد (Protocol كما تعلمنا في الفصل 15)
from typing import Protocol

class Database(Protocol):
    def query(self, sql: str) -> list: ...

class UserRepository:
    def __init__(self, db: Database):  # نعتمد على أي شيء يطابق الواجهة
        self.db = db

    def get_user(self, user_id: int):
        return self.db.query(f"SELECT * FROM users WHERE id={user_id}")

# الآن يمكن تمرير MySQL أو PostgreSQL أو حتى قاعدة بيانات وهمية للاختبار
repo = UserRepository(db=MySQLDatabase())
repo_test = UserRepository(db=FakeDatabaseForTesting())
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**الفائدة العملية الكبرى:** هذا التصميم يجعل **الاختبار** (الفصل القادم) أسهل بكثير — يمكنك تمرير قاعدة بيانات وهمية (Mock) للاختبار بدلاً من الاتصال بقاعدة بيانات حقيقية في كل اختبار.

> **💡 نصيحة خبير:** مبادئ SOLID ليست قواعد صارمة يجب تطبيقها دائمًا وفي كل مكان — تطبيقها المفرط على مشروع صغير قد يُعقّده دون داعٍ (يُسمى هذا أحيانًا Over-engineering). **استخدمها كأدوات تفكير** عند الشعور أن كودًا معيّنًا أصبح صعب الصيانة أو التوسيع، وليس كقائمة تحقق إلزامية لكل سطر تكتبه.

## 24.7 ممارسات إضافية للكود النظيف

### DRY: لا تُكرّر نفسك (Don't Repeat Yourself)

```python
# ❌ تكرار -- نفس المنطق مكرر 3 مرات
def calculate_circle_area(radius):
    return 3.14159 * radius ** 2

def calculate_circle_circumference(radius):
    return 2 * 3.14159 * radius

def calculate_sphere_volume(radius):
    return (4/3) * 3.14159 * radius ** 3

# ✅ استخراج الثابت المشترك
import math  # استخدام math.pi بدلاً من كتابة الرقم يدويًا في كل مكان

def calculate_circle_area(radius: float) -> float:
    return math.pi * radius ** 2

def calculate_circle_circumference(radius: float) -> float:
    return 2 * math.pi * radius
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### KISS: أبقِه بسيطًا وغبيًا (Keep It Simple, Stupid)

```python
# ❌ معقّد دون داعٍ
def is_even(n: int) -> bool:
    return True if n % 2 == 0 else False

# ✅ بسيط ومباشر
def is_even(n: int) -> bool:
    return n % 2 == 0
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> **⚠️ تحذير: الذكاء المفرط عدو الوضوح**
> ```python
> # ❌ «ذكي» جدًا لكن غير مقروء
> result = [x for x in range(100) if not any(x % i == 0 for i in range(2, int(x**0.5)+1)) and x > 1]
>
> # ✅ أوضح، حتى لو أطول قليلاً
> def is_prime(n: int) -> bool:
>     if n < 2:
>         return False
>     return all(n % i != 0 for i in range(2, int(n**0.5) + 1))
>
> result = [x for x in range(100) if is_prime(x)]
> ```
> الكود «الذكي جدًا» الذي يتباهى بحل كل شيء في سطر واحد غالبًا ما يكون عبئًا على كل من يقرأه لاحقًا — بما فيهم أنت. **الوضوح دائمًا أهم من الإيجاز.**

### YAGNI: لن تحتاجه (You Aren't Gonna Need It)

لا تبني مرونة أو ميزات «قد تحتاجها مستقبلاً» دون حاجة فعلية حالية. هذا يُعقّد الكود دون فائدة حقيقية.

```python
# ❌ مبالغة: نظام إعدادات معقّد لميزة بسيطة قد لا تُستخدم أبدًا
class ConfigurableGreeter:
    def __init__(self, language="ar", formal=False, emoji_support=True,
                 max_length=100, custom_templates=None, plugins=None):
        # ... 20 سطرًا من المرونة غير المستخدمة فعليًا

# ✅ ابدأ بسيطًا، وأضف المرونة فقط عند الحاجة الفعلية
def greet(name: str) -> str:
    return f"مرحبًا، {name}!"
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

## 24.8 نموذج عملي: قبل وبعد (Refactoring كامل)

لنرَ مثالًا متكاملًا يُطبّق كل ما تعلمناه في هذا الفصل:

In [ ]:
# ===== قبل: كود يعمل لكنه غير نظيف =====
def proc(l):
    r = []
    for i in l:
        if i['type'] == 'A':
            if i['amt'] > 100:
                r.append({'id': i['id'], 'final': i['amt'] * 0.9})
            else:
                r.append({'id': i['id'], 'final': i['amt']})
        elif i['type'] == 'B':
            r.append({'id': i['id'], 'final': i['amt'] * 0.95})
    return r

In [ ]:
# ===== بعد: نظيف، واضح، وقابل للصيانة =====

TYPE_A = "A"
TYPE_B = "B"

TYPE_A_BULK_THRESHOLD = 100
TYPE_A_BULK_DISCOUNT = 0.9
TYPE_B_STANDARD_DISCOUNT = 0.95

def calculate_type_a_discount(amount: float) -> float:
    """حساب السعر النهائي لعناصر النوع A (خصم إضافي للكميات الكبيرة)."""
    if amount > TYPE_A_BULK_THRESHOLD:
        return amount * TYPE_A_BULK_DISCOUNT
    return amount

def calculate_type_b_discount(amount: float) -> float:
    """حساب السعر النهائي لعناصر النوع B (خصم ثابت)."""
    return amount * TYPE_B_STANDARD_DISCOUNT

def process_items(items: list[dict]) -> list[dict]:
    """معالجة قائمة عناصر وحساب السعر النهائي لكل منها حسب نوعه.

    Args:
        items: قائمة قواميس، كل منها يحتوي 'id', 'type', 'amt'.

    Returns:
        قائمة قواميس تحتوي 'id' و'final' (السعر النهائي بعد الخصم).
    """
    processed_items = []

    for item in items:
        if item["type"] == TYPE_A:
            final_price = calculate_type_a_discount(item["amt"])
        elif item["type"] == TYPE_B:
            final_price = calculate_type_b_discount(item["amt"])
        else:
            continue  # تخطَّ الأنواع غير المعروفة بدلاً من كسر البرنامج

        processed_items.append({"id": item["id"], "final": final_price})

    return processed_items

لاحظ الفرق الهائل في القابلية للفهم، رغم أن **المنطق الأساسي لم يتغيّر إطلاقًا**!

## 24.9 تمارين

### 🟢 مستوى مبتدئ

1. ثبّت `black` و `ruff` في مشروع تجريبي، وشغّلهما على كود به مشاكل تنسيق واضحة (مسافات عشوائية، علامات اقتباس مختلطة).
2. أعد تسمية هذه المتغيرات لتكون أوضح: `d`, `tmp`, `lst2`, `flag1`.
3. استخرج «الأرقام السحرية» في الكود التالي إلى ثوابت مُسمّاة:

In [ ]:
def calculate_shipping(weight):
    if weight > 20:
        return weight * 5.5
    return weight * 3.2

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
# 2
# d -> user_data
# tmp -> temporary_file_path
# lst2 -> filtered_products
# flag1 -> is_user_verified

# 3
HEAVY_PACKAGE_THRESHOLD_KG = 20
HEAVY_PACKAGE_RATE = 5.5
STANDARD_RATE = 3.2

def calculate_shipping(weight: float) -> float:
    if weight > HEAVY_PACKAGE_THRESHOLD_KG:
        return weight * HEAVY_PACKAGE_RATE
    return weight * STANDARD_RATE

</details>

### 🟡 مستوى متوسط

4. أعد هيكلة (refactor) هذه الدالة لتطبيق مبدأ المسؤولية الواحدة (قسّمها لعدة دوال أصغر):

In [ ]:
def handle_user_signup(email, password):
    if "@" not in email:
        raise ValueError("بريد غير صالح")
    if len(password) < 8:
        raise ValueError("كلمة مرور قصيرة")
    hashed = password[::-1]  # تشفير وهمي للتبسيط
    print(f"حفظ المستخدم: {email}")
    print(f"إرسال بريد ترحيبي إلى: {email}")
    return {"email": email, "password": hashed}

5. طبّق Guard Clauses على هذه الدالة لتقليل التداخل:

In [ ]:
def get_shipping_cost(order):
    if order is not None:
        if order.weight > 0:
            if order.destination == "local":
                return order.weight * 2
            else:
                return order.weight * 5
    return 0

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
# 4
def validate_email(email: str) -> None:
    if "@" not in email:
        raise ValueError("بريد غير صالح")

def validate_password(password: str) -> None:
    if len(password) < 8:
        raise ValueError("كلمة مرور قصيرة")

def hash_password(password: str) -> str:
    return password[::-1]  # تشفير وهمي للتبسيط

def save_user(email: str, hashed_password: str) -> None:
    print(f"حفظ المستخدم: {email}")

def send_welcome_email(email: str) -> None:
    print(f"إرسال بريد ترحيبي إلى: {email}")

def handle_user_signup(email: str, password: str) -> dict:
    validate_email(email)
    validate_password(password)
    hashed = hash_password(password)
    save_user(email, hashed)
    send_welcome_email(email)
    return {"email": email, "password": hashed}

# 5
def get_shipping_cost(order) -> float:
    if order is None:
        return 0
    if order.weight <= 0:
        return 0
    if order.destination == "local":
        return order.weight * 2
    return order.weight * 5

</details>

### 🔴 مستوى تحدي

6. صمّم نظام إشعارات (Notification System) يطبّق مبدأي **Open/Closed** و **Dependency Inversion**: يجب أن تكون إضافة قناة إشعار جديدة (مثل SMS أو Push Notification) ممكنة **دون تعديل أي كود قديم موجود**، وأن تعتمد الدالة الرئيسية على تجريد (Protocol أو ABC) بدلاً من تفاصيل ملموسة.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
from abc import ABC, abstractmethod

class NotificationChannel(ABC):
    """تجريد (Abstraction) لأي قناة إشعار."""

    @abstractmethod
    def send(self, message: str, recipient: str) -> None:
        ...

class EmailNotification(NotificationChannel):
    def send(self, message: str, recipient: str) -> None:
        print(f"📧 إرسال بريد إلى {recipient}: {message}")

class SMSNotification(NotificationChannel):
    def send(self, message: str, recipient: str) -> None:
        print(f"📱 إرسال رسالة نصية إلى {recipient}: {message}")

# إضافة قناة جديدة -- بدون أي تعديل على الكود القديم (مبدأ Open/Closed)
class PushNotification(NotificationChannel):
    def send(self, message: str, recipient: str) -> None:
        print(f"🔔 إرسال إشعار دفع إلى {recipient}: {message}")

class NotificationService:
    """تعتمد على التجريد فقط -- مبدأ Dependency Inversion."""

    def __init__(self, channels: list[NotificationChannel]):
        self.channels = channels

    def notify_all(self, message: str, recipient: str) -> None:
        for channel in self.channels:
            channel.send(message, recipient)

# الاستخدام
service = NotificationService(
    channels=[EmailNotification(), SMSNotification(), PushNotification()]
)
service.notify_all("طلبك في الطريق!", "ahmad@example.com")

</details>

## 24.10 اختبر فهمك (Quiz)

<details>
<summary>1. ما الفرق بين وظيفة `black` ووظيفة `ruff`؟</summary>

`black` أداة تنسيق (Formatter) تهتم بـ**شكل** الكود (مسافات، علامات اقتباس، تقسيم الأسطر) وتُعدّله تلقائيًا بأسلوب موحّد وثابت. `ruff` أداة فحص (Linter) تهتم بـ**جودة ومنطق** الكود، فتكتشف أخطاء محتملة مثل متغيرات غير مستخدمة، استيراد زائد، ومخالفات أسلوبية لا يُصلحها التنسيق وحده.
</details>

<details>
<summary>2. ما الفرق بين التعليق الجيد والتعليق غير الضروري؟</summary>

التعليق غير الضروري يشرح «ماذا يفعل الكود» رغم أن الكود نفسه واضح بما يكفي لفهم ذلك. التعليق الجيد يشرح «لماذا» اتُّخذ قرار معيّن — معلومة سياقية لا يمكن استنتاجها من قراءة الكود وحده (كسبب استخدام قيمة معيّنة، أو قيد تقني خارجي).
</details>

<details>
<summary>3. اشرح مبدأ المسؤولية الواحدة (SRP) بمثال من حياتك اليومية بعيدًا عن البرمجة.</summary>

مثال: سكين المطبخ مسؤول فقط عن التقطيع، وليس عن الطبخ أو الغسيل أيضًا. إذا كانت أداة واحدة «تفعل كل شيء»، يصبح إصلاحها أو تحسينها لمهمة واحدة محفوفًا بمخاطر التأثير على المهام الأخرى. بالمثل، الصنف الذي يتعامل مع البيانات ويحفظها ويرسل بريدًا إلكترونيًا في آنٍ واحد يصعب صيانته وتعديله بأمان.
</details>

<details>
<summary>4. ما الفرق بين مبدأ DRY ومبدأ YAGNI؟</summary>

DRY («لا تُكرّر نفسك») يحثّ على استخراج المنطق المتكرر إلى مكان واحد بدلاً من نسخه في عدة أماكن. YAGNI («لن تحتاجه») يحثّ على عدم بناء مرونة أو ميزات إضافية لحالات استخدام مستقبلية افتراضية لم تظهر الحاجة الفعلية لها بعد. كلاهما يهدف لتبسيط الكود، لكن DRY يركّز على التكرار الحالي، وYAGNI يركّز على التعقيد المستقبلي غير الضروري.
</details>

<details>
<summary>5. لماذا يُعتبر الاعتماد على Protocol/ABC (التجريد) أفضل من الاعتماد المباشر على صنف ملموس محدد (مبدأ DIP)؟</summary>

لأن الاعتماد على تجريد يسمح باستبدال التفاصيل الملموسة بسهولة (كتغيير قاعدة بيانات، أو استخدام نسخة وهمية للاختبار) دون تعديل الكود الذي يستخدم ذلك التجريد. هذا يجعل الكود أكثر مرونة وأسهل اختبارًا، لأنه لا يكون «متجمّدًا» على تفصيل تقني واحد.
</details>

## 24.11 ملخص الفصل (Cheat Sheet)

```bash
# ===== الأدوات =====
uv add --dev black ruff

uv run black .                  # تنسيق تلقائي لكل الملفات
uv run black --check .          # فحص دون تعديل (لـ CI/CD)

uv run ruff check .             # فحص المشاكل المنطقية والأسلوبية
uv run ruff check --fix .       # إصلاح تلقائي لما يمكن إصلاحه
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

In [ ]:
# ===== قواعد تسمية PEP 8 =====
# متغيرات/دوال   -> snake_case
# ثوابت          -> UPPER_SNAKE_CASE
# أصناف          -> PascalCase
# خاص            -> _single_underscore / __double_underscore

| المبدأ | الفكرة بإيجاز |
|---|---|
| **S**RP | صنف واحد = مسؤولية واحدة |
| **O**CP | وسّع بإضافة كود جديد، لا بتعديل القديم |
| **L**SP | الصنف الفرعي يجب أن يحترم عقد الصنف الأساسي |
| **I**SP | واجهات صغيرة ومتخصصة، لا واجهات ضخمة شاملة |
| **D**IP | اعتمد على التجريد، لا على التفاصيل الملموسة |
| DRY | لا تُكرّر نفسك |
| KISS | أبقِه بسيطًا |
| YAGNI | لا تبنِ ما لا تحتاجه فعليًا الآن |

## 24.12 ماذا بعد؟

لقد تعلّمت الآن كيف تُميّز بين «كود يعمل» و«كود احترافي» — فرقٌ جوهري يُعرّف مسيرتك المهنية كمبرمج. لكن يبقى سؤال حاسم: **كيف تتأكد أن كودك «يعمل» فعلاً، وأنه سيستمر في العمل بشكل صحيح حتى بعد تعديله لاحقًا؟**

الجواب هو **الاختبار الآلي (Automated Testing)**. في الفصل القادم، سنتعلّم مكتبة **`pytest`** — المعيار الذهبي للاختبار في بايثون — لكتابة اختبارات تتحقق تلقائيًا من صحة كودك، تكتشف الأخطاء قبل أن يراها المستخدم، وتمنحك الثقة الكاملة لتعديل الكود دون خوف من كسر شيء ما دون أن تعلم.

---

### الخطوة التالية

- [الفصل الخامس والعشرون: الاختبار (Testing) باستخدام pytest](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-25.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الرابع والعشرون — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*